In [6]:
import subprocess
import json
import time
from youtube_transcript_api import YouTubeTranscriptApi, FetchedTranscript
from pathlib import Path

# ── Configuration ──────────────────────────────────────────────
CHANNEL_URL = "https://www.youtube.com/@SelfMadeMillennial"  # Replace with target channel URL
OUTPUT_DIR = Path("transcripts")                       # Folder to save transcript files
SLEEP_SECONDS = 1                                      # Delay between requests to avoid rate limiting


In [2]:
OUTPUT_DIR.mkdir(exist_ok=True)  # Create output folder if it doesn't exist

print(f"Fetching video list from: {CHANNEL_URL}")

Fetching video list from: https://www.youtube.com/@SelfMadeMillennial


In [3]:
result = subprocess.run(
    [
        "yt-dlp",
        "--flat-playlist",   # Only fetch metadata, do not download videos
        "--dump-json",       # Output each video as a JSON object
        CHANNEL_URL
    ],
    capture_output=True,
    text=True
)

In [4]:
# Parse each line of output as a separate JSON object (one per video)
videos = [json.loads(line) for line in result.stdout.strip().split("\n") if line]
print(f"Found {len(videos)} videos\n")

Found 370 videos



In [7]:
# ── Step 2: Loop through each video and extract transcript ──────
ytt = YouTubeTranscriptApi()

for index, video in enumerate(videos, start=1):
    video_id = video["id"]
    title = video.get("title", video_id)

    # Sanitise title for use as a filename (remove special characters)
    safe_title = "".join(c for c in title if c.isalnum() or c in " -_")[:80]
    output_file = OUTPUT_DIR / f"{safe_title}.txt"

    print(f"[{index}/{len(videos)}] {title}")

    # Skip if transcript file already exists (useful when re-running the script)
    if output_file.exists():
        print(f"  → Already exists, skipping\n")
        continue

    try:
        # Fetch English transcript using the YouTube Transcript API
        transcript = ytt.fetch(video_id, languages=["en"])

        # Join all transcript segments into a single block of text
        text = " ".join([entry["text"] for entry in transcript])

        # Write transcript to file with title and video ID as header
        with open(output_file, "w", encoding="utf-8") as f:
            f.write(f"Title: {title}\n")
            f.write(f"Video ID: {video_id}\n")
            f.write(f"URL: https://www.youtube.com/watch?v={video_id}\n\n")
            f.write(text)

        print(f"  → Saved to {output_file}\n")

    except Exception as e:
        # If transcript is unavailable (e.g. no captions), log and continue
        print(f"  → Skipped: {e}\n")

    # Wait between requests to avoid being rate limited by YouTube
    time.sleep(SLEEP_SECONDS)

[1/370] Why HR is One of the Smartest Careers You Can Choose Right Now
  → Skipped: 
Could not retrieve a transcript for the video https://www.youtube.com/watch?v=jB9NXW1mMCw! This is most likely caused by:

YouTube is blocking requests from your IP. This usually is due to one of the following reasons:
- You have done too many requests and your IP has been blocked by YouTube
- You are doing requests from an IP belonging to a cloud provider (like AWS, Google Cloud Platform, Azure, etc.). Unfortunately, most IPs from cloud providers are blocked by YouTube.

There are two things you can do to work around this:
1. Use proxies to hide your IP address, as explained in the "Working around IP bans" section of the README (https://github.com/jdepoix/youtube-transcript-api?tab=readme-ov-file#working-around-ip-bans-requestblocked-or-ipblocked-exception).
2. (NOT RECOMMENDED) If you authenticate your requests using cookies, you will be able to continue doing requests for a while. However, YouTube w

KeyboardInterrupt: 